# Level 10: Practice exercises

Try each one yourself first. Solutions are at the bottom of this notebook.

1. **`@debug`**: print the function name, arguments and return value (like `log_calls`), and use `functools.wraps`.
2. **`@slow_down(seconds)`**: sleep `seconds` before calling the function.
3. **`@ensure_positive`**: raise `ValueError` if any numeric positional argument is negative.
4. **`@cache_with_expiry(seconds)`**: like `memoize`, but a cached value expires after `seconds`.
5. **`@rate_limit(calls, period)`**: allow at most `calls` calls per `period` seconds, otherwise raise `RuntimeError`.
6. **Challenge: `@timeit(runs=N)`**: must work as both `@timeit` and `@timeit(runs=5)`. Run the function `runs` times and print the average time.

In [ ]:
import functools
import time
import random

In [ ]:
# Exercise 1: @debug

In [ ]:
# Exercise 2: @slow_down(seconds)

In [ ]:
# Exercise 3: @ensure_positive

In [ ]:
# Exercise 4: @cache_with_expiry(seconds)

In [ ]:
# Exercise 5: @rate_limit(calls, period)

In [ ]:
# Exercise 6: @timeit / @timeit(runs=5)

---
# Solutions

Scroll down only after you have tried.

<br><br><br><br><br><br><br><br><br><br>

In [ ]:
# Exercise 1
def debug(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"{func.__name__}(args={args}, kwargs={kwargs}) -> {result!r}")
        return result
    return wrapper

@debug
def area(w, h=1):
    return w * h
area(3, h=4)

# Exercise 2
def slow_down(seconds=1):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            time.sleep(seconds)
            return func(*args, **kwargs)
        return wrapper
    return decorator

@slow_down(0.2)
def countdown(n):
    return n - 1
print(countdown(3))

# Exercise 3
def ensure_positive(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        for a in args:
            if isinstance(a, (int, float)) and a < 0:
                raise ValueError(f"negative argument: {a}")
        return func(*args, **kwargs)
    return wrapper

@ensure_positive
def sqrt(x):
    return x ** 0.5
print(sqrt(16))
try:
    sqrt(-4)
except ValueError as e:
    print("Error:", e)

In [ ]:
# Exercise 4
def cache_with_expiry(seconds):
    def decorator(func):
        cache = {}
        @functools.wraps(func)
        def wrapper(*args):
            now = time.monotonic()
            if args in cache:
                value, stored_at = cache[args]
                if now - stored_at < seconds:
                    return value
            value = func(*args)
            cache[args] = (value, now)
            return value
        return wrapper
    return decorator

@cache_with_expiry(0.3)
def get_price(item):
    print(f"  (fetching {item}...)")
    return random.randint(1, 100)

print(get_price("apple"))
print(get_price("apple"))   # cached, no fetch
time.sleep(0.35)
print(get_price("apple"))   # expired, fetches again

# Exercise 5
def rate_limit(calls, period):
    def decorator(func):
        history = []
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            now = time.monotonic()
            while history and now - history[0] > period:
                history.pop(0)
            if len(history) >= calls:
                raise RuntimeError("rate limit exceeded")
            history.append(now)
            return func(*args, **kwargs)
        return wrapper
    return decorator

@rate_limit(calls=2, period=1)
def send_sms():
    return "sent"

print(send_sms(), send_sms())
try:
    send_sms()
except RuntimeError as e:
    print("Error:", e)

# Exercise 6
def timeit(func=None, *, runs=1):
    def decorator(f):
        @functools.wraps(f)
        def wrapper(*args, **kwargs):
            total = 0.0
            for _ in range(runs):
                start = time.perf_counter()
                result = f(*args, **kwargs)
                total += time.perf_counter() - start
            print(f"{f.__name__}: avg {total / runs:.6f}s over {runs} run(s)")
            return result
        return wrapper
    return decorator if func is None else decorator(func)

@timeit
def build_list():
    return [i for i in range(100_000)]

@timeit(runs=5)
def build_set():
    return {i for i in range(100_000)}

build_list(); build_set();

---
# Summary: the cheat sheet

```python
import functools

# 1. Simple decorator
def deco(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

# 2. Decorator with arguments (3 layers)
def deco_args(arg):
    def deco(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            return func(*args, **kwargs)
        return wrapper
    return deco

# 3. Optional arguments: @deco and @deco(x=1) both work
def deco_opt(func=None, *, x=1):
    def deco(f):
        @functools.wraps(f)
        def wrapper(*args, **kwargs):
            return f(*args, **kwargs)
        return wrapper
    return deco if func is None else deco(func)
```

**Remember:**
- `@d` is the same as `f = d(f)`, and `@d(x)` is the same as `f = d(x)(f)`.
- The decorator body runs once, at definition time. The wrapper runs on every call.
- Always use `*args, **kwargs`, always `return` the result, always use `functools.wraps`.
- Stacked decorators apply bottom-up and run top-down.
- Use a class-based decorator when you need a lot of state, and add `__get__` if it must work on methods.

**Next topic ideas:** generators and `yield`, context managers (`with`), iterators, `*args`/`**kwargs` in depth.